In [ ]:
import sys
sys.path.insert(1, os.path.abspath('..'))  # repo root

import importlib
import SMLM ; importlib.reload(SMLM)
from SMLM import *


import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pims
import os
from glob import glob
from dcimg import DCIMGFile
import scipy.optimize as opt
from skimage.io import imsave
import time
import sklearn.cluster
biplane = False

#
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple, Union
import warnings
import copy
import ipywidgets
import itertools
import math
import traitlets
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple, Union
from IPython.display import display
#from sdt import changepoint, flatfield, image, io, loc, multicolor, nbui, roi
from tqdm.notebook import tqdm
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.widgets as mwidgets
#import trackpy
import pickle
import tifffile
#import seaborn as sns

from scipy.optimize import curve_fit
from scipy.optimize import leastsq

from tqdm.notebook import tnrange, tqdm
from tqdm.contrib import tzip

#trackpy.quiet()

In [ ]:
cam = Camera(pixsize = 6500, 
             QE = 0.75,
             baseline = 100, 
             amp = 0.21, 
             EM_gain = 1,  
             readnoise = 0.7) 

obj = Objective(M = 44.83,
                NA = 1.,
                focal_len = 1.8e6,
                RI = 1.518,
                Ts_coefs =[-0.075, 0, -0.0694,  0, 1],
                Tp_coefs = [-0.0548, 0, 1])

In [ ]:
# laod images
filename = f'1000LLS_aber179_deskew.tif'
file = sorted(glob(f'{filename}'))
vid = [pims.open(n) for n in file]
raw_images = np.asarray(vid[0])
raw_images.shape

In [ ]:
stack = []
frame_start = 0
frame_end = vid[0].shape[0] 
row_start = 0
row_end = vid[0].shape[1]
col_start = 0
col_end = vid[0].shape[2]

img = raw_images[frame_start : frame_end, row_start : row_end, col_start : col_end]
# stack.append(cts2phs(img, cam, file_format='tif'))
stack.append(img)
stack[0] = np.maximum(stack[0], 1)

stack[0].shape

In [ ]:
ii = 0
vmin = np.min(stack[0][ii])
vmax = np.max(stack[0][ii])
print(vmin, vmax)
plt.imshow(stack[0][ii], cmap=parula)
plt.colorbar()
plt.show()

In [ ]:
# preloc setup
ll = 0
ii = 0
sigma = 1
min_dist = 7
threshold_abs = 1
threshold_rel = 100
N_crop = 19
figsize = (14, 4)

preloc_setup(stack[ll][ii], sigma=sigma, min_dist=min_dist, threshold_abs=threshold_abs, threshold_rel=threshold_rel, N_crop=N_crop, figsize=figsize)

In [ ]:
images, prelocs = [], []
for i in range(len(stack)) :
    print('Stack', i+1)
    tmp_prelocs, tmp_I_photons = preloc_and_crop(stack[i], sigma=sigma, min_dist = min_dist, threshold_abs=threshold_abs, threshold_rel=threshold_rel, N_crop =N_crop, show_progress=True)
    images.append(tmp_I_photons)
    prelocs.append(tmp_prelocs)
    show_prelocs(images[i], n=16)

In [ ]:
show_prelocs(images[ll], n=24)

In [ ]:
plt.imshow(images[0][100], cmap=parula)
plt.colorbar()
images[0][0].min()

In [ ]:
# MATLAB aberr
file = '1800LLS_aber179_aberrations_bead1_run2.mat'
noll_index, aberr, T_fit_coefs = load_aberr(file)
noll_index = noll_index.astype(int)
aberr[2] = 0 # delete defocus (Noll 4)

# mask = np.where(np.isin(noll_index, [5, 6, 7, 8, 9, 10, 11, 22, 37, 56]))
mask = np.where(np.isin(noll_index, np.arange(5, 67)))
noll_index = noll_index[mask]
aberr = aberr[mask]

noll_index, aberr

In [ ]:
import importlib
import SMLM ; importlib.reload(SMLM)
from SMLM import *

ll = 0
z_min = 0
z_max = 2000
h1 = psf(z_defocus = 1000,
        z_min = z_min,
        z_max = z_max,
        wavelength = 670,
        Nx = 19 + 4,
        os = 2,
        ux = cam.pixsize/obj.M,
        uz = 100,
        RI = [1.33, 1.33], # sample, immersion medium
        Z_modes = noll_index,
        Z_magn =  aberr,
        Ts_coefs = obj.Ts_coefs,
        Tp_coefs = obj.Tp_coefs,
        sigma = 100,
        interp_method = 'lanczos5',
        correct_RI_mismatch = 'no'
        )

h1.show_slice_xy()

In [ ]:
plt.imshow(h1.data[..., -1], cmap=parula)
plt.colorbar()

In [ ]:
h1.show_projection()
h1.show_pupil()
h1.show_CRLB(cam, s=2000, bg=10)

In [ ]:
sl = slice(len(images[ll]))
results, raw_results = perform_fit(images[ll][sl], prelocs[ll][sl], h1, show_progress=True)
show_results_2d(results, marker_size = 0.5)

In [ ]:
#FILTERING & VISUALIZATION
#show_results_3d(results)
#show_results_2d(results_r, marker_size = 0.2)
show_results_hist(results, raw_results, bins = 50)
# show_results_1d(results, raw_results, marker_size = 0.01)
show_results_time(results, raw_results, marker_size=0.01)

In [ ]:
# FILTERING

import importlib
import SMLM ; importlib.reload(SMLM)
from SMLM import *

pxl = 1.5
filtered_results, filtered_raw_results, filter_idx = filter_results(results, raw_results,
            # x_range = (1950, 20500), y_range = (1500, 35000),
            x_pxl_range = (-pxl, pxl), y_pxl_range = (-pxl, pxl),
            z_range = (-500, 500),
            # signal_range = (140, 5000),
            # bg_range = (20, 40),
            # LLR_range = (0, 3000),
            print_fraction=True)

# show_results_3d(filtered_results)
show_results_2d(filtered_results, marker_size = 0.1)
show_results_hist(filtered_results, filtered_raw_results, bins = 100)
show_results_time(filtered_results, filtered_raw_results, exp_time = 20, marker_size = 0.1)

In [ ]:
show_pixelation(h1, results, bins_plotted = 1, bins=20)

In [ ]:
show_molecules(images[0], filter_idx)

In [ ]:
# df = ts_results
df = filtered_results
fig = px.scatter(x = df['x'], y = df['y'], color = df['z'], opacity = 1)
fig.update_traces(marker_size=5)
# fig.update_layout(width=800, height=800, yaxis = dict(zeroline=True, autorange='reversed'))
fig.update_layout(width=800, 
                  height=800,
                  xaxis=dict(zeroline=False, showgrid=False, showticklabels=False),
                  yaxis=dict(zeroline=False, autorange='reversed', showgrid=False, showticklabels=False), 
                )
fig.show()

In [ ]:
# clusterization

data = ts_results.to_numpy()

method = 'DBSCAN'
method = 'DBSCAN3D'
# method = 'OPTICS'
# method = 'HDBSCAN'

# dbscan: larger eps implies smaller densities
eps = 80
min_samples = 40

#optics
max_eps = 200
xi = 0.05
min_cluster_size = 100

n_bins = 250
figsize = (18,18)
label = False
dict_nanorulers, data_npcs = find_npcs(data, method, eps, max_eps, xi, min_cluster_size, min_samples, label, n_bins, figsize)

In [ ]:
import importlib
import SMLM ; importlib.reload(SMLM)
from SMLM import *

if __name__ == '__main__':
    app = dash_plot(data_npcs)
    # app = dash_plot(filtered_results)
    app.run_server(debug=True, port=8052)